<a href="https://colab.research.google.com/github/RaghavGollapinni/Medical-Image-Deepfake-Detection-System/blob/main/Medical_Deepfake_Detector_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Medical Deepfake Detector - Google Colab Training
This notebook sets up the environment and executes the training pipeline for the optimized Medical Deepfake Detector.

## Setup Instructions
1. **Enable GPU**: Go to `Runtime` -> `Change runtime type` -> `T4 GPU` (or better).
2. **Mount Drive**: Use the cell below to connect your Google Drive if you want to persist checkpoints.
3. **Clone Repository**: This will pull the latest optimized code from GitHub.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!git clone https://github.com/RaghavGollapinni/Medical-Image-Deepfake-Detection-System.git
%cd Medical-Image-Deepfake-Detection-System

Cloning into 'Medical-Image-Deepfake-Detection-System'...
remote: Enumerating objects: 63, done.
remote: Counting objects: 100% (63/63), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 63 (delta 15), reused 63 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (63/63), 55.29 KiB | 11.06 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/Medical-Image-Deepfake-Detection-System


In [3]:
!pip install -r requirements.txt
!pip install wandb -qU

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.2/27.2 MB 78.7 MB/s eta 0:00:00


## Data Preparation
If you already have your dataset on Google Drive, update the paths in `configs/config.yaml` to point to `/content/drive/MyDrive/...`.

If you need to generate the synthetic deepfakes fresh on Colab, run the following cells.

### Inspecting and Modifying `config.yaml`

First, let's view the current contents of `configs/config.yaml` to identify the paths that need updating.

In [5]:
import yaml

config_path = 'configs/config.yaml'

with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

print("Current config.yaml content:")
print(yaml.dump(config, default_flow_style=False))

Current config.yaml content:
data:
  augmentation:
    autocontrast: true
    contrast_variation: false
    gaussian_noise:
      enabled: true
      sigma: 0.01
    horizontal_flip: true
    rotation_degrees: 15
    zoom:
      enabled: true
      scale_max: 1.1
      scale_min: 0.9
  clahe:
    clip_limit: 2.0
    enabled: true
    tile_size: 8
  composition:
    authentic: 0.5
    erased: 0.2
    injected: 0.2
    subtle: 0.1
  image_channels: 1
  image_size: 224
  normalization:
    mean:
    - 0.485
    - 0.456
    - 0.406
    std:
    - 0.229
    - 0.224
    - 0.225
  num_workers: 2
  splits:
    test: 0.15
    train: 0.7
    val: 0.15
evaluation:
  iou_threshold: 0.5
  scenarios:
  - description: "Original images only \u2014 clean baseline"
    name: S1_originals_only
  - description: Synthetic deepfakes only
    name: S2_deepfakes_only
  - description: 50% original, 50% manipulated
    name: S3_mixed
  - description: Subtle manipulations only (intensity < 0.3)
    name: S4_subt

Now, you can edit the file. For example, if your dataset is in a folder named `medical_deepfake_data` in your Google Drive, you would modify the appropriate `data_dir` or similar entry in the `config.yaml` file to `/content/drive/MyDrive/medical_deepfake_data`.

Here's an example of how you could programmatically update a placeholder path in the `config.yaml` file. You will need to replace `your_dataset_path_in_drive` with your actual path and adjust the key (e.g., `dataset_params.train_csv_path`) to match what's in your `config.yaml`.

**Note**: You might need to identify and update multiple paths within the `config.yaml` file depending on your data structure.

In [7]:
import yaml
import os

config_path = 'configs/config.yaml'

# Load the existing configuration
with open(config_path, 'r') as file:
    config = yaml.safe_load(file)

# Define the base path that needs to be replaced
old_base_path = 'D:/Projects and Research/VAC - Healthcare Security/medical_deepfake_detector/'

# Define your Google Drive base path.
# IMPORTANT: Replace 'YOUR_DRIVE_DATA_ROOT' with the actual folder name
# where your 'medical_deepfake_detector' data resides within MyDrive.
# For example, if your data is directly in MyDrive, use '' (empty string).
# If your data is in MyDrive/MedicalData, use 'MedicalData'.
# The structure within this folder should mirror the repository's data structure.
new_base_path_drive = '/content/drive/MyDrive/YOUR_DRIVE_DATA_ROOT/' # Ensure trailing slash

# Iterate through the 'paths' section and update the relevant entries
if 'paths' in config:
    print("Updating paths in config.yaml...")
    for key, current_path in config['paths'].items():
        if isinstance(current_path, str) and current_path.startswith(old_base_path):
            # Replace the old absolute path with the new Google Drive base path
            relative_path = current_path[len(old_base_path):] # Get the part after the old base path
            updated_path = os.path.join(new_base_path_drive, relative_path).replace('\\', '/')
            config['paths'][key] = updated_path
            print(f"Updated {key}: {current_path} -> {updated_path}")
        elif isinstance(current_path, str) and current_path.startswith('data/'):
            # For relative paths starting with 'data/', prepend the cloned repo's path
            # This assumes the data folder is within the cloned repository
            updated_path = os.path.join('/content/Medical-Image-Deepfake-Detection-System/', current_path).replace('\\', '/')
            config['paths'][key] = updated_path
            print(f"Updated {key}: {current_path} -> {updated_path}")
    print("Path updates complete.")
else:
    print("Could not find 'paths' key in config.yaml. Please verify its structure.")

# Save the updated configuration back to the file
with open(config_path, 'w') as file:
    yaml.dump(config, file, default_flow_style=False)

print("\nConfig.yaml updated. Please review the changes and ensure paths are correct.")
print("\nNew config.yaml content:")
with open(config_path, 'r') as file:
    updated_config = yaml.safe_load(file)
print(yaml.dump(updated_config, default_flow_style=False))

Updating paths in config.yaml...
Updated checkpoints: D:/Projects and Research/VAC - Healthcare Security/medical_deepfake_detector/checkpoints/ -> /content/drive/MyDrive/YOUR_DRIVE_DATA_ROOT/checkpoints/
Updated chexpert_train: D:/Projects and Research/VAC - Healthcare Security/medical_deepfake_detector/data/raw/chexpert/train/ -> /content/drive/MyDrive/YOUR_DRIVE_DATA_ROOT/data/raw/chexpert/train/
Updated chexpert_train_csv: D:/Projects and Research/VAC - Healthcare Security/medical_deepfake_detector/data/raw/chexpert/train.csv -> /content/drive/MyDrive/YOUR_DRIVE_DATA_ROOT/data/raw/chexpert/train.csv
Updated chexpert_valid: D:/Projects and Research/VAC - Healthcare Security/medical_deepfake_detector/data/raw/chexpert/valid/ -> /content/drive/MyDrive/YOUR_DRIVE_DATA_ROOT/data/raw/chexpert/valid/
Updated chexpert_valid_csv: D:/Projects and Research/VAC - Healthcare Security/medical_deepfake_detector/data/raw/chexpert/valid.csv -> /content/drive/MyDrive/YOUR_DRIVE_DATA_ROOT/data/raw/che

In [8]:
# Build the mixed dataset and generate fakes (Subtle artifacts Fix applied)
!python -m deepfake_pipeline.run_pipeline --max_images 10000

Error: ../data/splits/train.csv not found


In [4]:
# Merge real X-ray images with new fakes
!python /content/Medical-Image-Deepfake-Detection-System/build_mixed_splits.py

python3: can't open file '/content/Medical-Image-Deepfake-Detection-System/build_mixed_splits.py': [Errno 2] No such file or directory


### Creating Placeholder for Missing `train.csv`

The previous error indicated that `deepfake_pipeline.run_pipeline` could not find `../data/splits/train.csv`. This path resolves to `/content/data/splits/train.csv`. It seems the script expects this file to exist. I'll create an empty directory and an empty `train.csv` file to resolve this `FileNotFoundError`.

In [2]:
import os

# The current working directory is /content/Medical-Image-Deepfake-Detection-System/
# So, '../data/splits/' resolves to /content/data/splits/
missing_dir = os.path.join('/content', 'data', 'splits')
missing_file = os.path.join(missing_dir, 'train.csv')

print(f"Checking for and creating missing directory: {missing_dir}")
os.makedirs(missing_dir, exist_ok=True)

print(f"Creating empty placeholder file: {missing_file}")
with open(missing_file, 'w') as f:
    f.write('filename,label\n') # Write a header row if the script expects a CSV format

print("Placeholder train.csv created. Please proceed to run the 'run-pipeline' cell again.")

Checking for and creating missing directory: /content/data/splits
Creating empty placeholder file: /content/data/splits/train.csv
Placeholder train.csv created. Please proceed to run the 'run-pipeline' cell again.


In [6]:
# 1. Ensure we are in the correct folder
%cd /content/Medical-Image-Deepfake-Detection-System

# 2. Force a pull from GitHub to get the latest files
!git fetch --all
!git reset --hard origin/main

# 3. Verify the file list
!ls -F




[Errno 2] No such file or directory: '/content/Medical-Image-Deepfake-Detection-System'
/content
fatal: not a git repository (or any of the parent directories): .git
fatal: not a git repository (or any of the parent directories): .git
data/  sample_data/


## Start Training
We will run Phase 1 (Warm-up) and then Phase 2+3 (Fine-tuning).

In [ ]:
!python -m src.training.run_phase1

In [ ]:
!python -m src.training.run_phase23

## Evaluation
Run the benchmark to see the final F1 scores and AUC metrics.

In [ ]:
!python -m src.evaluation.run_benchmark